# Compute per-site band quality baseline
* Downloads (if not already present) the 30 min low-tide, 0% cloud satellite images per site across 2016-2026
* Computes the per-site, per-band median/MAD baseline (see sentinel2.compute_and_save_band_quality_baseline)
* Only needs to be run once - the baseline is cached to utils.get_band_quality_baseline_path() and reused by sentinel2.flag_anomalous_band_dates

In [1]:
import geopandas
import pathlib
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:63110,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:63139,Total threads: 2
Dashboard: http://127.0.0.1:63140/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:63114,


# Standardise council sites

In [3]:
data_path = utils.get_data_path()
council_site_path = data_path / "site_polygons" / "council_sites" / "standardised"

council_sites = {"AshleyRiver_Estuary.zip/AshleyRiver_Estuary": "Ashley Mouth ECAN",
                 "Brookland_Lagoon.zip/Brookland Lagoon": "Brookland Lagoon ECAN",
                 "porirua_intertidal_extent_GW.zip": "Porirua Estuary GWRC",
                 "Nelson Haven from OSM.geojson": "Nelson Haven NCC",
                 "Manukau_estuary_polygon_MHWS.zip/Manukau_estuary_polygon_MHWS": "Manukau Estuary AC"}

for key, value in council_sites.items():
    if not (council_site_path / f"{value}.gpkg").exists():
        print(f"Save {value}")
        if "zip" in key:
            zip_path = council_site_path.parent / key
            boundary = geopandas.read_file(f"/vsizip/{zip_path.as_posix()}")
        else:
            boundary = geopandas.read_file(council_site_path.parent / key)
        boundary.to_crs(utils.CRS_NZTM).to_file(council_site_path / f"{value}.gpkg")
    else:
        print(f"Already created {value}")

Already created Ashley Mouth ECAN
Already created Brookland Lagoon ECAN
Already created Porirua Estuary GWRC
Already created Nelson Haven NCC
Already created Manukau Estuary AC


# Values to edit

In [4]:
baseline_sites = list(council_sites.values())
baseline_tide_deltas = {"30mins": {"hrs": 0, "mins": 30},
                        "45mins": {"hrs": 0, "mins": 45},
                        "1hr": {"hrs": 1, "mins": 0}}
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [5]:
data_path = utils.get_data_path()
utils.create_data_folders()

# Cells to run
* Download the low-tide no-cloud images by year and site (skips sites already downloaded)
* Compute and save the per-site, per-band quality baseline

### Download satellite images
The 0% cloud, low-tide images across sites for each year and tide delta (30min/45min/1hr),
reusing the prediction-style per-year storage. Pooling across tide deltas (rather than dropping
the low-tide requirement entirely) grows the per-site sample size while keeping the baseline
restricted to genuinely low-tide reflectance.

In [6]:
satellite_paths = {}
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    _, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        use_council_paths=True)
    satellite_paths[tide_delta_name] = satellite_path

In [7]:
satellite_paths

{'30mins': WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_30mins_cloud_0/satellite_images'),
 '45mins': WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_45mins_cloud_0/satellite_images'),
 '1hr': WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_1hrs_cloud_0/satellite_images')}

In [8]:
baseline_sites

['Ashley Mouth ECAN',
 'Brookland Lagoon ECAN',
 'Porirua Estuary GWRC',
 'Nelson Haven NCC',
 'Manukau Estuary AC']

In [9]:
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    _, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        satellite_only=True,
        use_council_paths=True) 

    for baseline_site in baseline_sites[-1:]:
        for year in years:
            print(f"{tide_delta_name} {baseline_site}: year {year}")

            satellite_file = satellite_path / f"{baseline_site}_{year}_sentinel-2.nc"
            site_polygon_file = council_site_path / f"{baseline_site}.gpkg"

            if (satellite_path / f"{baseline_site}_{years[-1]}_sentinel-2.nc").exists():
                print(f"\tAlready run for {baseline_site} as {baseline_site}_{years[-1]}_sentinel-2.nc exists. "
                      "Delete if you want redownloaded.")
                break

            if not satellite_file.exists():
                print(f"\tSave satellite image of the site {baseline_site} around lowtide without cloud in the year {year}")
                site_polygon = geopandas.read_file(site_polygon_file)
                for i in range(3):
                    try:
                        satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                            geometry=site_polygon,
                            max_cloud_cover=baseline_max_cloud_cover,
                            year=year,
                            low_tide_delta_hrs=tide_delta["hrs"],
                            low_tide_delta_mins=tide_delta["mins"],
                            anomaly_filter_baseline=None,
                        )
                        if len(satellite_data['time']) == 0:
                            print("Warning: No satellite data without cloud. Ignore")
                        else:
                            print("\t\tSave")
                            utils.write_netcdf_conventions_in_place(satellite_data)
                            satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                            satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                            utils.write_netcdf_conventions_in_place(satellite_data)
                            utils.save_netcdf(satellite_data, satellite_file)
                        break
                    except Exception:
                        print("Caught exception downloading images and try again.")
                

30mins Manukau Estuary AC: year 2016
	Save satellite image of the site Manukau Estuary AC around lowtide without cloud in the year 2016
	Tide API-filtered tiles: 4 from 158 STAC items
	Tide API-filtered tiles: 1 from 158 STAC items
	Low cloud tiles: 0 from the low tide tiles of 1. Cloud percentages [39.8] Kept [False]
30mins Manukau Estuary AC: year 2017
	Save satellite image of the site Manukau Estuary AC around lowtide without cloud in the year 2017
	Tide API-filtered tiles: 8 from 187 STAC items
	Tide API-filtered tiles: 2 from 187 STAC items
	Low cloud tiles: 0 from the low tide tiles of 2. Cloud percentages [77.4 16.9] Kept [False False]
30mins Manukau Estuary AC: year 2018
	Save satellite image of the site Manukau Estuary AC around lowtide without cloud in the year 2018
	Tide API-filtered tiles: 20 from 289 STAC items
	Tide API-filtered tiles: 5 from 289 STAC items
	Low cloud tiles: 0 from the low tide tiles of 5. Cloud percentages [ 5.7 19.7 52.1 21.3  9.6] Kept [False False Fal

### Compute and save the band quality baseline
Pools all downloaded years across all tide deltas per site into one dataset, then computes
per-site, per-band median/MAD and caches it to utils.get_band_quality_baseline_path().

In [10]:
baseline = sentinel2.compute_and_save_council_band_quality_baseline(
    council_sites=baseline_sites,
    satellite_images_paths=list(satellite_paths.values()),
    training_baseline_file=utils.get_band_quality_baseline_path(),
)
baseline

	WARNING - no baseline satellite images for site Manukau Estuary AC under [WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_30mins_cloud_0/satellite_images'), WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_45mins_cloud_0/satellite_images'), WindowsPath('C:/Local/repos/seagrass-detection/data/council_predictions/tide_1hrs_cloud_0/satellite_images')]. Skipping this site.
	Saved council band quality baseline to C:\Local\repos\seagrass-detection\data\council_band_quality_baseline.csv


,site,band,median,mad,mad_raw,n_images,reference_sha256
0,Ashley Mouth ECAN,B04,904.0,69.245098,60.5,31,385dc0774adc288d394acac002b515ed08f39bbea67f85...
1,Ashley Mouth ECAN,B03,846.0,48.177079,27.0,31,385dc0774adc288d394acac002b515ed08f39bbea67f85...
2,Ashley Mouth ECAN,B02,668.0,38.950752,15.0,31,385dc0774adc288d394acac002b515ed08f39bbea67f85...
3,Brookland Lagoon ECAN,B04,736.0,58.086957,48.0,49,385dc0774adc288d394acac002b515ed08f39bbea67f85...
4,Brookland Lagoon ECAN,B03,731.0,56.855522,47.0,49,385dc0774adc288d394acac002b515ed08f39bbea67f85...
5,Brookland Lagoon ECAN,B02,572.0,60.398382,54.0,49,385dc0774adc288d394acac002b515ed08f39bbea67f85...
6,Porirua Estuary GWRC,B04,596.0,68.650000,54.5,20,385dc0774adc288d394acac002b515ed08f39bbea67f85...
7,Porirua Estuary GWRC,B03,690.0,74.000775,67.0,20,385dc0774adc288d394acac002b515ed08f39bbea67f85...
8,Porirua Estuary GWRC,B02,542.0,71.287209,66.5,20,385dc0774adc288d394acac002b515ed08f39bbea67f85...
9,Nelson Haven NCC,B04,948.0,62.033333,20.5,10,385dc0774adc288d394acac002b515ed08f39bbea67f85...


In [11]:
import pandas
pandas.read_csv(r"C:\Local\repos\seagrass-detection\data\band_quality_baseline_tide_api.csv")

,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,502.00,51.494618,8.00,16
1,Ihutai,B03,642.00,59.765278,31.00,16
2,Ihutai,B02,568.00,53.913715,29.00,16
3,CatlinsLake,B04,356.00,57.154375,45.50,50
4,CatlinsLake,B03,378.00,63.650714,56.00,50
5,CatlinsLake,B02,262.00,52.527054,44.00,50
6,CatlinsRiverMouth,B04,814.00,93.410750,96.00,55
7,CatlinsRiverMouth,B03,877.00,85.140667,86.00,55
8,CatlinsRiverMouth,B02,637.00,50.491917,42.00,55
9,Childrens,B04,423.00,82.479365,82.00,159
